##**Projeto:** Merca Data Platform

##**Squad:** 2 | Streaming em Tempo Real
### Objetivo
Monitorar continuamente o ADLS e processar automaticamente novos snapshots assim que forem detectados, gravando os dados no SQL Server.
### O que este notebook faz
| Etapa | Descrição |
| 1 | Carrega os snapshots já existentes como "já processados" (evita reprocessamento no boot) |
| 2 | Entra em loop e, a cada `INTERVALO_SEG` segundos, verifica novos snapshots |
| 3 | Para cada snapshot novo: lê os Parquets de cada tabela e grava no SQL Server |
| 4 | Erros em um ciclo são logados mas não interrompem o polling |
### Configurações do Polling
| Parâmetro | Valor padrão | Descrição |
| `POLLING_INFINITO` | `False` | `True` para produção contínua; `False` limita a `MAX_CICLOS` |
| `MAX_CICLOS` | `10` | Número máximo de ciclos quando `POLLING_INFINITO = False` |
| `INTERVALO_SEG` | `30` | Segundos de espera entre cada verificação |
| `TABELAS_MONITORAR` | `TABELAS_SQUAD2` | Tabelas verificadas em cada ciclo |
### Tabelas processadas
- `ecommerce_categorias`
- `ecommerce_itens_pedido`
- `ecommerce_produtos`
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | `listar_snapshots`, `ler_parquet`, `gravar_sql`, funções de log |
> **Nota:** O modo atual usa polling (verificação periódica a cada 30s).
> Quando a infraestrutura de event-driven estiver disponível, substituir por
> agendamento near-real-time via Azure Event Hubs ou similar.


In [0]:
%run ../utils/feat_squad2_99_helpers

- Configurações do Polling
Ajuste `POLLING_INFINITO = True` ao mover para produção.
Em desenvolvimento, use `POLLING_INFINITO = False` com `MAX_CICLOS` limitado para evitar execuções indefinidas no cluster.

In [0]:
# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO POLLING
# TODO: Substituir POLLING_INFINITO = True por
#       agendamento near real-time quando
#       infraestrutura estiver disponível
# ─────────────────────────────────────────────

POLLING_INFINITO  = True   # False = usa MAX_CICLOS
MAX_CICLOS        = 10     # usado apenas se POLLING_INFINITO = False
INTERVALO_SEG     = 30     # segundos entre verificações
TABELAS_MONITORAR = TABELAS_SQUAD2

log.info("Configurações do Polling:")
log.info(f"  Modo     : {'Infinito' if POLLING_INFINITO else f'Limitado ({MAX_CICLOS} ciclos)'}")
log.info(f"  Intervalo: {INTERVALO_SEG}s")
log.info(f"  Tabelas  : {TABELAS_MONITORAR}")


- Funções do Polling
**`processar_snapshot()`** — Para um snapshot novo detectado, lê o Parquet de cada tabela e grava no SQL Server em modo `append`. Erros por tabela são isolados e não bloqueiam as demais.
**`log_resultado()`** — Exibe o resumo de sucesso/erro por tabela após cada snapshot processado.


In [0]:
# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO POLLING
# TODO: Substituir POLLING_INFINITO = True por
#       agendamento near real-time quando
#       infraestrutura estiver disponível
# ─────────────────────────────────────────────

POLLING_INFINITO  = False   # True quando for para produção
MAX_CICLOS        = 10     # usado apenas se POLLING_INFINITO = False
INTERVALO_SEG     = 30     # segundos entre verificações
TABELAS_MONITORAR = TABELAS_SQUAD2

log.info("Configurações do Polling:")
log.info(f"  Modo     : {'Infinito' if POLLING_INFINITO else f'Limitado ({MAX_CICLOS} ciclos)'}")
log.info(f"  Intervalo: {INTERVALO_SEG}s")
log.info(f"  Tabelas  : {TABELAS_MONITORAR}")

In [0]:
# ─────────────────────────────────────────────
# FUNÇÕES DO POLLING
# ─────────────────────────────────────────────

def processar_snapshot(snapshot_id: str) -> dict:
    """
    Processa todas as tabelas de um snapshot novo.
    Lê os parquet e grava no SQL Server.

    Args:
        snapshot_id: caminho do snapshot ex: 2026/04/27/225222

    Returns:
        dict: resultado do processamento por tabela
    """
    resultado = {}

    for tabela in TABELAS_MONITORAR:
        try:
            # Lê do ADLS
            df = ler_parquet(snapshot_id, tabela)

            # Grava no SQL Server
            sucesso = gravar_sql(df, tabela, mode="append")

            resultado[tabela] = {
                "status" : "OK" if sucesso else "ERRO",
                "linhas" : df.count()
            }

        except Exception as e:
            log.error(f"Erro ao processar {tabela}: {str(e)}")
            resultado[tabela] = {
                "status": "ERRO",
                "erro"  : str(e)
            }

    return resultado


def log_resultado(snapshot_id: str, resultado: dict) -> None:
    """Loga o resultado do processamento de um snapshot."""
    log.info(f"Resultado snapshot {snapshot_id}:")
    for tabela, info in resultado.items():
        if info["status"] == "OK":
            log.info(f"  OK {tabela} → {info['linhas']} linhas")
        else:
            log.error(f"  ERRO {tabela} → {info.get('erro', 'erro desconhecido')}")


- Loop de Polling
O loop inicializa carregando todos os snapshots já existentes como "já processados", garantindo que apenas dados **novos** (chegados após o início do notebook) sejam ingeridos.
A cada ciclo:
1. Lista os snapshots atuais no ADLS
2. Calcula a diferença em relação aos já processados
3. Processa os novos em ordem cronológica
4. Aguarda `INTERVALO_SEG` segundos antes do próximo ciclo
O notebook pode ser interrompido manualmente com `KeyboardInterrupt` sem gerar erro.


In [0]:
# ─────────────────────────────────────────────
# LOOP DE POLLING
# ─────────────────────────────────────────────

inicio = log_inicio("feat_squad2_01_polling_ingestao")

# Carrega snapshots existentes como já processados
snapshots_processados = listar_snapshots()
total_processados     = 0
ciclo                 = 0

log.info(f"{len(snapshots_processados)} snapshot(s) existentes ignorados.")
log.info("Aguardando novos snapshots...\n")

try:
    while POLLING_INFINITO or ciclo < MAX_CICLOS:
        ciclo += 1
        agora  = datetime.now().strftime("%H:%M:%S")

        log.info(f"[{agora}] Ciclo {ciclo} — verificando novos snapshots...")

        try:
            # Lista snapshots atuais
            snapshots_atuais = listar_snapshots()

            # Detecta novos
            novos = snapshots_atuais - snapshots_processados

            if novos:
                log.info(f"  {len(novos)} novo(s) snapshot(s) detectado(s)!")

                for snapshot_id in sorted(novos):
                    resultado = processar_snapshot(snapshot_id)
                    log_resultado(snapshot_id, resultado)
                    snapshots_processados.add(snapshot_id)
                    total_processados += 1

            else:
                log.info(" Nenhum snapshot novo encontrado.")

        except Exception as e:
            # Erro em um ciclo não interrompe o polling
            log.error(f"Erro no ciclo {ciclo}: {str(e)}")
            log.info("Continuando polling...")

        # Aguarda antes do próximo ciclo
        log.info(f"Aguardando {INTERVALO_SEG}s...")
        time.sleep(INTERVALO_SEG)

except KeyboardInterrupt:
    # Permite interrupção manual sem erro
    log.info("Polling interrompido manualmente.")

finally:
    log.info(f"Total de snapshots processados: {total_processados}")
    log_fim("feat_squad2_01_polling_ingestao", inicio)